# Progressive Growing Generative Adversarial Network (ProGAN) Applied to Women Clothes Dataset

Source:<P>

https://blog.paperspace.com/implementation-of-progan-from-scratch/

Adapted:<P>

Antonio Esteves @ UMinho, May 2024<P>

ProGAN was propsed in the paper [Progressive Growing of GANs for Improved Quality, Stability, and Variation](https://arxiv.org/pdf/1710.10196.pdf) to generate high-quality images. In this notebook, we will implement ProGAN using PyTorch. The implementation follows closely the original paper.

The dataset that we will use in this notebook is [the Women clothes dataset](https://www.kaggle.com/datasets/tauilabdelilah/women-clothes?select=clothes) from Kaggle, which contains 16240 upper clothes for women with 256x192 resolution. It is a small dataset with low resolution compared to the one that the authors of ProGAN use which contains 800k images with high resolution 1024x1024 but it still gives us good results.

## Import the necessary libraries

We will import
* `torch` since we will use PyTorch,
* `torch.nn` to model the GAN,
* `torch.optim`, a package that implements various optimization algorithms (e.g. sgd, adam,etc),
* `torchvision.datasets` and `torchvision.transforms` to prepare the data and apply some transformations,
* `functional` as `F` from `torch.nn` to upsample the images using `interpolate`,
* `DataLoader` and `Dataset` from `torch.utils.data` to create mini-batches,
* `save_image` from `torchvision.utils` to save some fake samples,
* `log2` form `math` because we need the inverse representation of the power of 2 to implement the adaptive minibatch size depending on the output resolution,
* `numpy` for linear algebra,
* `os` for interaction with the operating system,
* `tqdm` e `trange` from `tqdm.notebook` to show progress bars,
* `matplotlib.pyplot` to show the results and compare them with the real ones.
* `Path` from `pathlib` to handle file paths.
* `natsorted`from `natsort` to sort the files in a folder.
* `Image` from `PIL` to handle images.
* `wandb`for session monitoring and logging.

In [ ]:
import torch
from   torch               import nn, optim
from   torchvision         import datasets, transforms
import torch.nn.functional as     F
from   torch.utils.data    import DataLoader, Dataset
from   torchvision.utils   import save_image, make_grid
from   math                import log2
import numpy               as     np
import os
from   tqdm.notebook       import trange, tqdm
import wandb
import matplotlib.pyplot   as     plt

from   pathlib             import Path
from   natsort             import natsorted
from   PIL                 import Image

import wandb
import yaml
import time

## Seed everything

Let us seed everything to make the results reproducible.

In [ ]:
def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## Read the configuration file

* Initialize `dataset_path` with the path of the real images.
* Specify the start train at image size with 4x4, as the original paper.
* Initialize the `device` to `“cuda"` if it is available and `“cpu”` otherwise.
* Set the learning rate to `0.001`.
* The batch size will be depend on the resolution of the images that we want to generate in each step, so we initialize `batch_sizes` with a list of values, but this list can be adjusted to our GPU VRAM.
* Initialize `image_size` with 128.
* Set `channels_image` to 3 because we will generate 128 by 128 RGB images.
* In the original paper, they initialize `z_dim` and `in_channels` with 512, but we initialize them with 256 to use less memory and speed-up training. We could perhaps even get better results if we double these hyperparameters.
* With ProGAN, we can use any of the GANs loss functions we want but since we will follow the paper decisions, we will use the same loss function as they used, the Wasserstein loss function, also known as WGAN-GP, and proposed in the paper [Improved Training of Wasserstein GANs](https://arxiv.org/pdf/1704.00028.pdf). This loss includes the parameter $\lambda$, which is commonly set to 10.
* Initialize `progressive_epochs` with 30, which defines the number of opochs we run with each image size.

* `sample_interval`, `int`,   default=400,    Interval between image generations during training.
* `log_interval`,    `int`,   default=100,    Interval between successive logs.
* `experiment_name`, `str`,              ,    Base name of the files created by the notebook.
* `dataset`,         `str`,              ,    Dataset used for training the models.

In [ ]:
CONFIG_FILE = 'config/config_progan_02.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

In [ ]:
data_path = Path(config["dataset_path"])
train_dir = data_path / "train"

print(torch.__version__)

# setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

## Login into W&B

In [ ]:
wandb.login()

## Track metadata and hyperparameters with `wandb.init`

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(project='GAN_CLOTHES', entity='ajesteves', config=config_wandb)

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

## Create a custom Dataset from the images in a folder and the associated attributes using `torch.utils.data.Dataset`

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc = os.path.join(self.root_dir, self.total_images[idx])
        image   = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

## i. Define the transformation that will be applied to the images

- resize the images to the resolution that we want,
- convert the images to tensors,
- apply some augmentations,

## ii. Instantiate a Custom Dataset

## iii. Create a training DataLoader

- Select the current batch size from the list `batch_sizes`, using as index `int(log2(image_size/4)`.
- This is actually how we implement a mini-batch size that depends on the images resolution.

In [ ]:
def get_loader(image_size, dataset_train_dir):

    train_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Normalize(
                [0.5 for _ in range(config["channels_image"])],
                [0.5 for _ in range(config["channels_image"])],
            )
        ]
    )
    batch_size = config["batch_sizes"][int(log2(image_size / 4))]

    train_data = CustomDataSet(
        root_dir  = dataset_train_dir,
        transform = train_transform,
    )

    train_loader = DataLoader(
        dataset     = train_data,
        batch_size  = batch_size,
        shuffle     = True,
        num_workers = 4,
        pin_memory  = True,
    )

    return train_loader, train_data


Let us check if everything works fine and display a few real images.

In [ ]:
def check_dataloader(image_size, dataset_train_dir, nrows, ncols):

    NT       = nrows * ncols
    loader,_ = get_loader(config["image_size"], dataset_train_dir)

    nreads = 0
    while nreads < NT:
        batch  = next(iter(loader))
        bs     = batch.shape[0]
        if nreads == 0:
            cloth = batch
        else:
            cloth = torch.cat((cloth, batch), dim = 0)
        nreads += bs

    _, ax    = plt.subplots(nrows, ncols, figsize=(3*ncols,3*nrows))
    plt.suptitle('Some real images of cloth', fontsize=15, fontweight='bold')

    index = 0
    for r in range(nrows):
        for c in range(ncols):
            if nrows==1:
                ax[c].imshow((cloth[index].permute(1,2,0)+1)/2) 
            else:
                ax[r][c].imshow((cloth[index].permute(1,2,0)+1)/2)
            index += 1

In [ ]:
check_dataloader(image_size=config["image_size"], dataset_train_dir=train_dir, nrows=5, ncols=5)

## Model implementation

Now let us implement the ProGAN generator and discriminator as described in the original paper. 
Specifically, we have to imaplements:

- Progressive growing (of model and layers)
- Mini-batch standard deviation on discriminator
- Pixel normalization
- Equalized learning rate

Let us begin by building the generator.<P>

![Generator](fig/progan_generator_arch.png)

In the figure above, we can see the architecture of the generator.<P>
The number of channels will be 256, instead of 512, in 4 growing steps, then we decrease the number of channels to 1/2, then to 1/4, etc.<P>
Let us define a variable containing the factors that will be used, in the generator and discriminator, to multiply the number of channels and obtain the value that will be used in each layer.

In [ ]:
factors = [1, 1, 1, 1, 1/2, 1/4, 1/8, 1/16, 1/32]

## Equalized Learning Rate

Now let us implement equalized learning rate for the generator. We will name the class as `WSConv2d`, weighted scaled convolutional layer, which inherits `nn.Module`.

- The arguments of the `_init_` method are `in_channels`, `out_channels`, `kernel_size`, `stride`, and `padding`. We use them to create a normal convolutional layer.
- Then, we define a scale that will be $\sqrt{\frac{2}{k*k*ch}}$, where $k$ is the filter size and $ch$ is the number of channels/features.
- Next, we copy the `bias` of the current layer into a variable because we do not want to scale the bias of the convolution layer; then we remove the bias, and finally, we initialize the convolutional layer weights with random normal distributed values, i.e., from N(0,1).
- In the `forward` method, we pass `x * scale` through the convolution and we add the `bias` after reshaping it.

In [ ]:
class WSConv2d(nn.Module):
    '''
    Weighted scaled convolutional layer.
    The weights of the Conv2d are initialized with values from N(0,1) and then 
    these values are divided by scale=sqrt(2/(filtersize*filtersize*inchannels)).
    The bias is not scaled and it added to the Conv2d output:
       Conv2d(input*scale)+bias
    '''
    def __init__(
        self, in_channels, out_channels, kernel_size=3, stride=1, padding=1,
        ):
        super(WSConv2d, self).__init__()
        self.conv      = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.scale     = (2 / (in_channels * (kernel_size ** 2))) ** 0.5
        self.bias      = self.conv.bias # Copy the bias of the current column layer
        self.conv.bias = None           # Remove the bias

        # Initialize the convolutional layer
        nn.init.normal_(self.conv.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x):
        return self.conv(x * self.scale) + self.bias.view(1, self.bias.shape[0], 1, 1)

## Pixel Normalization

Now let us create a `PixelNorm` class, for pixel normalization.<P>
In the `_init_` method we define epsilon as $10^{−8}$﻿.<P>
In the `forward` method, we normalize `x` using the expression:
$$x'_{i,j} = \frac{x_{i,j}}{\sqrt{\frac{1}{F} \sum_{f=0}^{F-1} (x^f_{i,j}})^2 + \epsilon}$$

In [ ]:
class PixelNorm(nn.Module):
    '''
    Pixel normalization class.
    Divides each pixel value by "the square root of mean squared value" of all pixels.
    '''
    def __init__(self):
        super(PixelNorm, self).__init__()
        self.epsilon = 1e-8

    def forward(self, x):
        return x / torch.sqrt(torch.mean(x ** 2, dim=1, keepdim=True) + self.epsilon)

## ConvBlock

The `Generator` architecture repeats 2 convolution layers with $3 \times 3$ filters several times, so let us write a class that is composed of the next layers and that makes the code more modular:
* Weighted scaled convolutional layer 1.
* Leaky ReLU activation function 1.
* An optional pixel normalization layer 1.
* Weighted scaled convolutional layer 2.
* Leaky ReLU activation function 2.
* An optional pixel normalization layer 2.
  
This ConvBlock will be used in the discriminator as well, the only difference between the two is that the discriminator will not use the pixel normalization layer.

- The arguments of the `_init_` method are: `in_channels`, `out_channels`, and `use_pixelnorm`.
- This method instantiates the layers of the ConvBlock: `conv1` of type `WSConv2d`, which maps `in_channels` to `out_channels`, `conv2` of type `WSConv2d` and maps `out_channels` to `out_channels`, `leaky` of type `LeakyReLU` with a slope of 0.2 as mentioned in the ProGAN paper, `pn` of type `PixelNorm`.
- The `forward` method defines the block computations on the forward pass:<P>
  x -> conv1 -> leaky -> (optional) pn -> conv2 -> leaky -> (optional) pn -> x'


In [ ]:
class ConvBlock(nn.Module):
    '''
    Convolutional block composed of the following layers:
    * Weighted scaled convolutional layer 1.
    * Leaky ReLU activation function 1.
    * An optional pixel normalization layer 1.
    * Weighted scaled convolutional layer 2.
    * Leaky ReLU activation function 2.
    * An optional pixel normalization layer 2.
    '''
    def __init__(self, in_channels, out_channels, use_pixelnorm=True):
        super(ConvBlock, self).__init__()
        self.use_pn = use_pixelnorm
        self.conv1  = WSConv2d(in_channels, out_channels)
        self.conv2  = WSConv2d(out_channels, out_channels)
        self.leaky  = nn.LeakyReLU(0.2)
        self.pn     = PixelNorm()

    def forward(self, x):
        x = self.leaky(self.conv1(x))
        x = self.pn(x) if self.use_pn else x
        x = self.leaky(self.conv2(x))
        x = self.pn(x) if self.use_pn else x
        return x

## Generator

We build now the generator.

- The `__init__` method instantiates the layers necessary to the generator:
  - The first block of layers in the `Generator` architecture is different from the other blocks. So, in the `__init__` method we instantiate a distinct block called `initial` with the following layers: `PixelNorm`, `ConvTranspose2d`, `LeakyReLU`, `WSConv2d`, `LeakyReLU`, and `PixelNorm`.
  - It is also instantiated a `initial_rgb` block that is a `WSConv2d` that maps `in_channels` to `img_channels` (3 for RGB).
  - Next, it is created the `prog_blocks` module that will contain the convolutional block, of the type `ConvBlock`, added in each progressive step to the architecture. The number of `ConvBlock`s included is equal the number of ProGAN progressive steps. The number of input and output channels in each `ConvBlock` vary according to defined multiplicative factors.
  - It is also created the `rgb_layers` module that will contain the `WSConv2d` layer added in each progressive step to the architecture. These blocks convert the input data to RGB (3 channels).

- The `fade_in` method implements the smooth fade in of the layers added in each progressive growing step.
  - This method has the arguments `alpha`, `scaled` and `generated`, and returns: 
  `tanh(alpha∗generated+(1−alpha)∗upscale)`
  - The reason why we use `tanh` is that it will be the output (the generated image) and we want the pixels to be in the range [-1:+1].

- The `forward` method has arguments
  - `z`, which has dimension `Z_dim`
  - `alpha`, which is used to implement the smooth fade in during training; `alpha` is between 0 and 1
  - `steps`, which is the number of the current progressive prowing step, and is used to know the resolution that we are working with: for `steps`=0 images are 4x4, for `steps`=1 images are 8x8,etc.
  - the method passes `z` through `initial` block, we check if `steps` = 0 and if it is, then all we want to do is run it through the initial toRGB block; otherwise, we loop over the number of steps, and in each step we upscale the current "image" (creates `upscaled`) and we run the current "image" through the ConvBlock that working on the current step resolution (create `out`);
  - In the end, we apply the `fade_in` method to `out` and `upscaled`, using the current value of `alpha`, after converting the "image" to RGB.

In [ ]:
class Generator(nn.Module):

    def __init__(self, z_dim, in_channels, img_channels=3):
        super(Generator, self).__init__()

        # The initial block converts a random vector with shape (z_dim x 1 x 1) to (z_dim x 4 x 4).
        self.initial = nn.Sequential(
            PixelNorm(),
            nn.ConvTranspose2d(z_dim, in_channels, 4, 1, 0),
            nn.LeakyReLU(0.2),
            WSConv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2),
            PixelNorm(),
        )

        self.initial_rgb = WSConv2d(
            in_channels, img_channels, kernel_size=1, stride=1, padding=0
        )

        self.prog_blocks, self.rgb_layers = (
            nn.ModuleList([]),
            nn.ModuleList([self.initial_rgb]),
        )

        for i in range(len(factors) - 1):  # -1 to prevent index error because of factors[i+1]
            conv_in_c  = int(in_channels * factors[i])
            conv_out_c = int(in_channels * factors[i + 1])
            self.prog_blocks.append(ConvBlock(conv_in_c, conv_out_c))
            self.rgb_layers.append(
                WSConv2d(conv_out_c, img_channels, kernel_size=1, stride=1, padding=0)
            )

    def fade_in(self, alpha, upscaled, generated):
        # alpha should be scalar within [0, 1], and upscale.shape == generated.shape
        return torch.tanh((1 - alpha) * upscaled + alpha * generated)

    def forward(self, z, alpha, steps):
        out = self.initial(z)

        # If we are in first progressive growing step,
        # this 'to RGB' layer is the generator output
        if steps == 0:
            return self.initial_rgb(out)

        # For each progressive step that was run before, applies:
        # * a upsampling operation and a ConvBlock layer
        for step in range(steps):
            # Upsampling operation relative to progressive growing 'step'
            upscaled = F.interpolate(out, scale_factor=2, mode="nearest")
            # ConvBlock relative to progressive growing 'step'
            out      = self.prog_blocks[step](upscaled) 

        # For the current progressive step, applies:
        # toRGB(last upsampling output) * (1-alpha) + toRGB(last ConvBlock output) * alpha

        # The number of channels in upscale will stay the same,
        # while 'out' (which passed through prog_blocks) might change. 
        # To ensure we can convert both to rgb, we use different rgb_layers
        # 'steps-1' and 'steps' for 'upscaled' and 'out', respectively.

        # toRGB(last upsampling output) 
        final_upscaled = self.rgb_layers[steps - 1](upscaled)

        # toRGB(last ConvBlock output)
        final_out      = self.rgb_layers[steps](out)

        # Fade in: tanh((1 - alpha) * final_upscaled + alpha * final_out)
        out            = self.fade_in(alpha, final_upscaled, final_out)

        return out

## Discriminator

Now, we will create the discriminator, which the WGAN-GP paper names by "critic" and we are using WGAN-GP.<P>

If we compare next table with the generator table, we conclude that the discriminator architecture roughly mirrors the generator architecture, except changing upsampling to downsampling, and they grow in synchrony during the progressive steps.<P>

![discriminator](fig/progan_discriminator_arch.png)

- The `__init__` method has the following arguments: `in_channels` and `img_channels`.
  - It initializes `leaky`, of type `LeakyReLu`, with a slope of 0.2.
  - Next, it is created the `prog_blocks` module that will contain the convolutional block, of the type `ConvBlock`, added in each progressive step to the architecture. The number of `ConvBlock`s included is equal the number of ProGAN progressive steps. The number of input and output channels in each `ConvBlock` vary according to defined multiplicative factors (the factors are used in the reverse order of the generator).
  - It is also created the `rgb_layers` module that will contain the `WSConv2d` layer added in each progressive step to the architecture. These blocks convert from RGB (3 channels) to the number of channels specific of each step.
  - Next, it instantiates the "final_rgb" layer, which is the fromRGB layer that works with 4x4 "images".
  - It instantiates a downsampling layer using average pooling.
  - It create a final block of layers with: a WSConv2d layer, a LeakyReLU activation, a second WSConv2d layer, a second LeakyReLU, and a third WSConv2d.

- The `fade_in` method implements the smooth fade in of the layers added in each progressive growing step.
  - This method has the arguments `alpha`, `downscaled` and `out`, and returns:
    $alpha∗out+(1−alpha)∗downscaled$

- The `minibatch_stddev` method implements a mini-batch standard deviation layer for the discriminator.
  - It calculates the stddev for each sample (across all channels, and pixels).
  - Then, we repeat the calculation for a single channel and concatenate the result with the image. 
  - In this way, the discriminator will get information about the variation in the images of a mini-batch.

- The `forward` method has these arguments:
  - `x` input "image";
  - `alpha`, which is used to implement the smooth fade in during training;
  - `steps`, which is the number of the current progressive growing step; the step used by the discriminator is 'total_steps-steps' (the mirror value of the value used by the generator);
  - The first operation of the discriminator is convert the image from RGB to `in_channels`, a values that depends on the image size. Then it applied a LeakyReLU activation function.

  - It checks if `steps=0`, and if it is we just use `minibatch_stddev` layer and the final block.
  - Otherwise, it applies the computations relative to the current progressive growing step.
  - The, it is applied the fade in of the layers added in current step.
  - For each progressive step that was run before, applies a ConvBlock layer and a downsampling operation.
  - It applies the mini-batch standard deviation layer.
  - Applies the `final_block` to produce the discriminator output.

In [ ]:
class Discriminator(nn.Module):

    def __init__(self, in_channels, img_channels=3):
        super(Discriminator, self).__init__()
        self.prog_blocks, self.rgb_layers = nn.ModuleList([]), nn.ModuleList([])
        self.leaky                        = nn.LeakyReLU(0.2)

        # In the discriminator, we create the progressive architecture from the end 
        # to the beginning, a strategy that mirrors what we did in the generator.
        #
        # In this way, we use 'factors' by starting at the end of the list to the beginning.
        # So the first ConvBlock and toRGB layer we append will work with an input of
        # size equal to 1024x1024, then 512x512, then 256x256, etc.

        for i in range(len(factors) - 1, 0, -1):
            conv_in  = int(in_channels * factors[i])
            conv_out = int(in_channels * factors[i - 1])
            self.prog_blocks.append(
                ConvBlock(conv_in, conv_out, use_pixelnorm=False)
            )
            self.rgb_layers.append(
                WSConv2d(img_channels, conv_in, kernel_size=1, stride=1, padding=0)
            )

        # The "initial_rgb" layer is the fromRGB layer that works with 4x4 "images"
        # and "mirrors" the generator 'initial_rgb'
        self.initial_rgb = WSConv2d(
            img_channels, in_channels, kernel_size=1, stride=1, padding=0
        )

        self.rgb_layers.append(self.initial_rgb)

        # Downsampling layer using average pooling
        self.avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

        # This is the block for "images" of size 4x4
        self.final_block = nn.Sequential(
            # +1 to in_channels because we concatenate from mini-batch stddev
            WSConv2d(in_channels + 1, in_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            WSConv2d(in_channels, in_channels, kernel_size=4, padding=0, stride=1),
            nn.LeakyReLU(0.2),
            # we use this 'WSConv2d' layer instead of the Linear layer indicated in the Table
            WSConv2d(in_channels, 1, kernel_size=1, padding=0, stride=1),
        )

    def fade_in(self, alpha, downscaled, out):
        """
        Used to fade in downscaled using the average pooling output 
        and the output from ConvBlock.
        """
        # alpha should be scalar within [0, 1], and upscale.shape == generated.shape
        return alpha * out + (1 - alpha) * downscaled

    def minibatch_stddev(self, x):
        batch_statistics = (
            torch.std(x, dim=0).mean().repeat(x.shape[0], 1, x.shape[2], x.shape[3])
        )
        # We calculate the stddev for each sample (across all channels, and pixels).
        # Then, we repeat the calculation on a single channel and concatenate the
        # the result with the image. In this way, the discriminator will get
        # information about the variation in the images of a mini-batch.
        return torch.cat([x, batch_statistics], dim=1)

    def forward(self, x, alpha, steps):
        # The step used by the discriminator is 'total_steps-steps',
        # the mirror value of the value used by the generator.
        cur_step = len(self.prog_blocks) - steps

        # Convert from RGB as initial step, the output channels used depend on
        # the image size, and apply a LeakyReLU activation function.
        out = self.leaky(self.rgb_layers[cur_step](x))

        if steps == 0:  # i.e, image is 4x4
            out = self.minibatch_stddev(out)
            return self.final_block(out).view(out.shape[0], -1)

        # Because 'prog_blocks' might change the channels, for downsampling we use rgb_layer
        # from previous/smaller size, which in our case corresponds to add 1 in the index.

        # Computations relative to the current progressive growing step
        # * x -> downsampling -> fromRGB layer[cur_step+1] -> downscaled
        # out -> ConvBlock -> Downsampling -> out
        downscaled = self.leaky(self.rgb_layers[cur_step + 1](self.avg_pool(x)))
        out = self.avg_pool(self.prog_blocks[cur_step](out))

        # The fade_in is done first between the downscaled and the input;
        # this is the opposite of what is done in the generator.
        #
        # out = alpha * out + (1 - alpha) * downscaled
        out = self.fade_in(alpha, downscaled, out)

        # For each progressive step that was run before, applies:
        # * a ConvBlock layer and a downsampling operation
        for step in range(cur_step + 1, len(self.prog_blocks)):
             # Downsampling operation relative to progressive growing 'step'
            out = self.prog_blocks[step](out)
            # ConvBlock relative to progressive growing 'step'
            out = self.avg_pool(out)

        # Applies the mini-batch standard deviation layer
        out = self.minibatch_stddev(out)

        # Applies the final_block to produce the discriminator output
        out = self.final_block(out).view(out.shape[0], -1)

        return out

## Utilities

In [ ]:
def gradient_penalty(discriminator, real, fake, alpha, train_step, device="cpu"):
    '''
    Implements the gradient penalty used in WGAN-GP loss.
    '''
    BATCH_SIZE, C, H, W = real.shape
    beta = torch.rand((BATCH_SIZE, 1, 1, 1)).repeat(1, C, H, W).to(device)
    interpolated_images = real * beta + fake.detach() * (1 - beta)
    interpolated_images.requires_grad_(True)

    # Calculate discriminator scores
    mixed_scores = discriminator(interpolated_images, alpha, train_step)

    # Take the gradient of the scores with respect to the images
    gradient = torch.autograd.grad(
        inputs=interpolated_images,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradient         = gradient.view(gradient.shape[0], -1)
    gradient_norm    = gradient.norm(2, dim=1)
    gradient_penalty = torch.mean((gradient_norm - 1) ** 2)
    return gradient_penalty

In [ ]:
def generate_images(generator, step, n=64):
    '''
    Function that given the 'generator' model, the number of steps to identify 
    the current resolution, and a number n=64, and generate 'n' fake images 
    and save them to file.
    '''
    base_file_name = f'results/{config["experiment_name"]}'
    generator.eval()
    alpha = 1.0

    with torch.inference_mode():
        noise = torch.randn(n, config["z_dim"], 1, 1).to(device)
        img   = generator(noise, alpha, step)
        if not os.path.exists(f'{base_file_name}/step{step}'):
            os.makedirs(f'{base_file_name}/step{step}')
        save_image(img*0.5+0.5, f"{base_file_name}/step{step}/img_{i}.png")

    generator.train()

def generate_grid_images(generator, step, grid_size=8, base_file_name='results/default_experiment'):
    '''
    Function that given the 'generator' model, the number of steps to identify 
    the current resolution, and the size of the grif of images to generate,
    it will generate grid_size*grid_size=64 fake images and save them to a file.
    It also uploads the file to w&B
    '''
    generator.eval()
    alpha = 1.0

    with torch.inference_mode():

        noise = torch.randn(grid_size**2, config["z_dim"], 1, 1).to(device)
        img   = generator(noise, alpha, step)

        img  = (img * 0.5 + 0.5)
        img  = torch.clamp(img, min=0.0, max=1.0)
        img  = (img * 255)
        img  = img.to('cpu', dtype=torch.uint8)
        grid = make_grid(img, nrow=grid_size, padding=1, pad_value=1)
        grid = grid.permute(1, 2, 0)
        grid = grid.numpy()

        # Upload the grid of images to W&B
        wandb_image = wandb.Image(grid, caption="ProGAN generated images")
        wandb.log({"ProGAN generated images": wandb_image})

        # Display the grid of images
        fig = plt.figure(figsize=(10, 10), constrained_layout=True)
        plt.imshow(grid)

        # Save the grid of images as a PNG file
        file_png = f"{base_file_name}_generated_step{step}.png"
        plt.imsave(file_png, grid)

    generator.train()

## Functions to save and load the models to/from file

In [ ]:
def save_model_and_results(generator, discriminator, results, hyperparameters, file_name):
    results_to_save = {
        'generator':       generator.state_dict(),
        'discriminator':   discriminator.state_dict(),
        'results':         results,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(generator, discriminator, file_name, config, device):
    '''
    Given instances of the generator and discriminator models, loads from file 'file_name':
    (i)   the weights of both models,
    (ii)  the results obtained during model training and
    (iii) the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    discriminator.load_state_dict(results_loaded['discriminator'])
    discriminator.to(device)

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['hyperparameters']

## Training the model

In this section, we will train the ProGAN.

First, let us use this line of code to get some additional performance benefits.

In [ ]:
torch.backends.cudnn.benchmarks = True

### Training function

First, we loop over all the mini-batches created by `dataloader`, and we take just the images because we do not need labels, then we identify the current batch size because we need it later.

Then we train the discriminator, where we want to maximize `E(discriminator(real)) - E(discriminator(fake))`. This expression quantifies the capacity of the discriminator to distinguish real from fake images. If it has a large value, it means that the discriminator identifies a large difference between real and fake images. If the value is zero, it means the discriminator cannot distinguish between real and fake images.

After that, we train the generator, where we want to maximize `E(discriminator(fake))`. Because the generator wants to fool the discriminator, if we maximize this term means that `E(discriminator(real)) - E(discriminator(fake))` will assume a smaller value, which is the opposite of what the discriminator wants.

Finally, we update the value of `alpha`, which is used in `fade_in` and ensure that it is between 0 and 1.

In [ ]:
def train_fn(
    discriminator,
    generator,
    dataloader,
    dataset,
    step,
    epoch,
    alpha,
    opt_disc,
    opt_gen,
    results,
    ):

    start_time = time.time()

    pbar = tqdm(dataloader, leave=True)

    # ..........................................
    # Loop over all the DataLoader batches
    # ..........................................
    for batch_idx, real in enumerate(pbar):
        real           = real.to(device)
        cur_batch_size = real.shape[0]

        # ........................................
        # Train the discriminator
        # ........................................

        # Loss used to train the discriminator:
        #       maximize  E[discriminator(real)] - E[discriminator(fake)]
        #   <=> minimize -E[discriminator(real)] + E[discriminator(fake)]

        noise       = torch.randn(cur_batch_size, config["z_dim"], 1, 1).to(device)

        fake        = generator(noise, alpha, step)
        disc_real   = discriminator(real, alpha, step)
        disc_fake   = discriminator(fake.detach(), alpha, step)
        gp          = gradient_penalty(discriminator, real, fake, alpha, step, device=device)

        loss_disc = (
            - (torch.mean(disc_real) - torch.mean(disc_fake))
            + config["lambda_gp"] * gp
            + (0.001 * torch.mean(disc_real ** 2)) )

        discriminator.zero_grad()
        loss_disc.backward()
        opt_disc.step()

        # ........................................
        # Train the generator
        # ........................................

        # Loss used to train the generator:
        #       maximize E[discriminator(gen_fake)]
        #   <=> minimize -E[discriminator(gen_fake)]

        gen_fake = discriminator(fake, alpha, step)
        loss_gen = -torch.mean(gen_fake)

        generator.zero_grad()
        loss_gen.backward()
        opt_gen.step()

        # Update alpha and ensure it is less than 1
        alpha += cur_batch_size / (
            (config["progressive_epochs"][step] * 0.5) * len(dataset)
        )
        alpha = min(alpha, 1)

        # Save the results in dictionary
        results["d_loss"].append(loss_disc.item())
        results["g_loss"].append(loss_gen.item())
        results["alpha"].append(alpha)

        pbar.set_postfix(
            gp=gp.item(),
            loss_disc=loss_disc.item(),
        )

        # Print progress metrics and save them to W&B
        if batch_idx % config["log_interval"] == 0:

            mean_d_loss = np.mean(results["d_loss"][-config["log_interval"]:])
            mean_g_loss = np.mean(results["g_loss"][-config["log_interval"]:])

            print(f'step|epoch|iter: {step :3d} | {epoch+1 :5d} | {batch_idx : 5d} / {len(dataloader) : 6d} ({(batch_idx*100)/len(dataloader) :0>5.1f}%)', end="    ")
            print(f'Discriminator loss: {mean_d_loss :0>13.10f}', end="    ")
            print(f'Generator loss: {mean_g_loss :0>13.10f}')

            try:
                # Log metrics to W&B
                wandb.log(
                    {
                    "discriminator_loss": mean_d_loss,
                    "generator_loss":     mean_g_loss,
                    "alpha":              alpha,
                    "epoch":              epoch+1,
                    "step":               step,
                    }
                )
            except Exception as ex:
                print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

    end_time  = time.time()
    texec_sec = end_time - start_time
    texec_str = time_format(texec_sec)

    print(f'Epoch training time: {texec_str}')

    results["epoch_training_time"] = texec_str

    wandb.log(
        {
        "epoch_training_time_sec": texec_sec,
        "epoch_training_time":     texec_str
         }
    )

    return alpha

## Training

Now since we have everything let us put them together to train the ProGAN.

We start by initializing the generator, the discriminator, and the optimizers in the same way that was done in the ProGAN paper.<P>
Then, we put the generator and the discriminator in training mode.<P>
Next, we loop over `progressive_epochs`, and in each loop, we train the model for a given number of epochs.<P>
Then, we generate some fake images and save them, as a result, using `generate_examples` function.<P>
Finally, we progress to the next image resolution.<P>

In [ ]:
# Create an empty dictionary to store the training results
results = {
    'd_loss': [],
    'g_loss': [],
    'alpha':  [],
    'epoch_training_time': 0.0,
}

# Initialize generator and discriminator according to WGAN paper
# (since it no longer outputs between [0, 1])
generator = Generator(
    config["z_dim"], 
    config["in_channels"],
    img_channels = config["channels_image"]
    ).to(device)

discriminator = Discriminator(
    config["in_channels"],
    img_channels = config["channels_image"]
    ).to(device)

# Initialize optimizers
opt_gen = optim.Adam(
    generator.parameters(),
    lr    = config["lr"],
    betas = (config["beta1"], config["beta2"])
)

opt_disc = optim.Adam(
    discriminator.parameters(),
    lr    = config["lr"],
    betas = (config["beta1"], config["beta2"])
)

# Put the models in training mode
generator.train()
discriminator.train()

step = int(log2(config["start_image_size"] / 4))

# ...............................................
# Loop relative to the progressive growing steps
# ...............................................

for num_epochs in config["progressive_epochs"]:

    # Start with very low alpha, but we can start with alpha=0
    alpha = 1e-5

    # 4->0, 8->1, 16->2, 32->3, 64 -> 4
    loader, dataset = get_loader(4 * 2 ** step,  dataset_train_dir=train_dir)
    print(f"Current image size: {4 * 2 ** step}")

    # .........................................................
    # Loop relative to the epochs of a progressive growing step
    # .........................................................

    for epoch in range(num_epochs):

        print(f"\nEpoch [{epoch+1}/{num_epochs}]")

        alpha = train_fn(
            discriminator,
            generator,
            loader,
            dataset,
            step,
            epoch,
            alpha,
            opt_disc,
            opt_gen,
            results
        )

    # generate_images(generator, step, n=64)

    generate_grid_images(
        generator,
        step,
        grid_size=8,
        base_file_name=f'results/{config["experiment_name"]}'
    )

    # Progress to the next image size
    step += 1

## Save the trained models to file

In [ ]:
file_save_model = f'models/{config["experiment_name"]}.pth'

save_model_and_results(
    generator,
    discriminator,
    results,
    config,
    file_save_model,
)

## Interpolation in the Latent Space and save interpolated images to a GIF file

In [ ]:
def interpolate(n, generator, step, z_dim, img_channels, img_size):

    # sample two noise vectors z1 and z2 from N(0,I)
    noise = torch.randn(2, z_dim, 1, 1).to(device)

    # define the step size and sample numbers in the range (0, 1) at
    # step intervals
    delta    = 1 / n
    _lambda_ = list(np.arange(0, 1, delta))

    # initialize a tensor for storing interpolated images
    interp_images = torch.zeros([n, img_channels, img_size, img_size])

    # put the model in evaluation mode
    generator.eval()

    # iterate over each value of _lambda_
    for i in range(n):

        # compute interpolated z
        z_interp = (1 - _lambda_[i]) * noise[0] + _lambda_[i] * noise[1]

        # generate the corresponding image
        with torch.inference_mode():
            gen_image = generator(z_interp, alpha=1.0, steps=step)
            interp_images[i] = gen_image

    # return the interpolated images
    return interp_images


In [ ]:
# call the interpolate function
interp_images = interpolate(
    16,
    generator,
    step         = 6,
    z_dim        = config["z_dim"],
    img_channels = config["channels_image"],
    img_size     = 256,
)

# visualize output images
grid = make_grid(
    interp_images.clamp(min=-1, max=1),
    scale_each=True,
    normalize=True,
)

plt.figure(figsize = (10, 10))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())

fname = f'results/{config["experiment_name"]}/{config["experiment_name"]}_interpolation.png'

# save visualizations
save_image(
    interp_images.clamp(min=-1, max=1),
    fname,
    nrow       = 4,    # number of images per row
    scale_each = True, 
    normalize  = True,
)

In [ ]:
# define a transform to convert a tensor to PIL image
transformT2I = transforms.ToPILImage()

imgs = []
for id in range(interp_images.shape[0]):
    # convert the tensor to PIL image using above transform
    imgs.append(transformT2I(interp_images[id]))


In [ ]:
# Make a GIF with interpolated images

fname = f'results/{config["experiment_name"]}/{config["experiment_name"]}_interpolation.gif'
gif = imgs[0]

gif.save(
    fname,
    format="GIF",
    append_images=imgs,
    save_all=True,
    duration=300, # duration of displaying each frame
    loop=0
)

## Results

In the figure below you can see the result that we obtain after training this ProGAN in this [dataset](https://www.kaggle.com/datasets/tauilabdelilah/women-clothes?ref=blog.paperspace.com) with 128*x 128 resolution.

![fake images](fig/progan_generated_imgs.png)

## Conclusion

In this article, we make a clean, simple, and readable implementation from scratch of ProGAN with the key attributions from the paper (progressive growing, fading-in new layers, mini-batch standard deviation on discriminator, normalization with PixelNorm, and equalized learning rate) using PyTorch.

In the upcoming articles, we will explain in depth and implement from scratch StyleGANs to generate also some cool fashion.

In [ ]:
# Mark the W&B run as finished
wandb.finish()